In [ ]:
# main.py
import random
import json
import re
import os
import time
from typing import List, Dict
from langchain_community.chat_models import ChatOllama
from langchain.prompts import ChatPromptTemplate
from persona import generate_persona, Persona
import requests

# 導入升級版指標計算與日誌服務
from simulationLoggerService import SimulationLoggerService, calculate_metrics

# === 0. 釋放記憶體服務 ===
def unload_ollama_model(model_name: str):
    """
    強迫卸載模型，並透過不斷嘗試或強制等待，確保 Ollama 釋放硬體。
    """
    url_generate = "http://localhost:11434/api/generate"
    url_chat = "http://localhost:11434/api/chat"
    payload = {"model": model_name, "keep_alive": 0}
    
    try:
        # 同時對 generate 和 chat 接口發送 keep_alive: 0，確保全方位覆蓋
        requests.post(url_generate, json=payload, timeout=5)
        requests.post(url_chat, json={"model": model_name, "messages": [], "keep_alive": 0}, timeout=5)
        print(f"\n[VRAM 釋放指令發送] 已通知 GPU 卸載模型: {model_name}")
    except Exception as e:
        print(f"[VRAM 釋放提示] 連線異常（可忽略）: {e}")
    
    # 回收 CUDA 記憶體
    print(f"[硬體冷卻] 正在等待 10 秒，確保 CUDA 完全釋放 {model_name}...")
    time.sleep(10)

# === 1. 實驗參數設定（完全對齊論文 6 人設計） ===
NUM_AGENTS = 6           # 論文標準：6 智慧體架構
SIMULATION_ROUNDS = 5    # 論文標準：5 輪次討論
REPETITIONS = 25         # 論文標準：獨立重複實驗 25 次
TOPIC = "關於數位身分證(eID)推行的隱私與便利性爭議"

# Group A 測試清單
GROUP_A_MODELS = [
    "llama3:8B",
    "qwen2.5:7b",
    "deepseek-r1:8b"
]

JUDGE_MODEL = "qwen2.5:32b-instruct"
STANCE_SCORE_MAP = {"Support": 9, "Neutral": 5, "Oppose": 2}


# === 2. 核心動態代理人初始化 ===
def initialize_matrix_simulation(model_name: str):
    agents = {}
    personas = {}

    # 論文基準控制組：2支持 / 2反對 / 2中立 (共6人)
    controlled_stances = ["Support", "Support", "Oppose", "Oppose", "Neutral", "Neutral"]
    random.shuffle(controlled_stances)

    for i in range(NUM_AGENTS):
        name = f"Agent_{i+1}"
        initial_stance = controlled_stances[i]
        p = generate_persona(name=name, initial_stance=initial_stance)
        personas[name] = p

        llm = ChatOllama(model=model_name, temperature=0.7)

        prompt_template = ChatPromptTemplate.from_messages([
            ("system", f"{p.to_prompt()}\n你現在正在參與一個網路討論區。請始終保持你的角色立場與說話風格。請直接像人類一樣發言，不要帶有任何 AI 的前言或系統贅字。"),
            ("human", "{input}"),
        ])
        agents[name] = prompt_template | llm

    return agents, personas


# === 3. 階段一：純對話流模擬 (高併發常駐 VRAM 模式) ===
def run_dialogue_pipeline(model_name: str, current_run: int) -> dict:
    """
    專注於對話生成，不混入 any Judge 評分，讓相同大小的模型全速跑完討論。
    """
    agents, personas = initialize_matrix_simulation(model_name)
    conversation_history = []

    print(f" -> [Dialogue Stage] Running {model_name} | Run #{current_run}/{REPETITIONS}")

    for r in range(SIMULATION_ROUNDS):
        agent_names = list(agents.keys())
        for name in agent_names:
            # 6人矩陣下，滑動視窗改為 -6，確保能看全上一輪所有人的最新發言
            context = "\n".join(conversation_history[-6:])

            # 🌟 提示詞優化：加強立場約束力，減少 Llama 3 開局角色扮演失效（投敵）的機率
            prompt = f"""
            目前的討論話題是：{TOPIC}。
            這是最近的討論進度：
            {context}
            
            請根據你的角色人格與初始立場，做出對應且自然的回應。像真實的人類在社交平台發言。
            """
            res_obj = agents[name].invoke({"input": prompt})
            response = str(res_obj.content).strip()

            # Deepseek-R1 <think> 標籤深度清洗
            if "<think>" in response and "</think>" in response:
                response = re.sub(r'<think>.*?</think>', '', response, flags=re.DOTALL).strip()
            elif "</think>" in response:
                response = response.split("</think>")[-1].strip()

            conversation_history.append(f"{name}: {response}")

    # 回傳這場實驗的元數據，供後續批次評估使用
    return {
        "run_id_hint": current_run,
        "personas": personas,
        "conversation_history": conversation_history
    }


# === 4. 階段二：批次判官評估流 (Batch Judge Pipeline) ===
def evaluate_batch_experiments(model_name: str, batch_data: List[dict], logger_service: SimulationLoggerService):
    """
    集中式批次評分。此時 32B Judge 模型一次載入，連續評估 25 場實驗。
    """
    print(f"\n" + "="*60)
    print(f"🚀 [Batch Judge Stage] Loading {JUDGE_MODEL} for bulk evaluation...")
    print("="*60)
    
    analyzer = ChatOllama(model=JUDGE_MODEL, temperature=0)

    # 第一次加載 32B 大模型時給予緩衝的flag
    is_first_judge_call = True

    for data in batch_data:
        run_id = data["run_id_hint"]
        personas = data["personas"]
        conversation_history = data["conversation_history"]

        print(f" Analyzing Run #{run_id}/{REPETITIONS}...")
        initial_scores = []
        final_scores = []
        agent_details = {}

        for name, persona in personas.items():
            # 🌟 修正 1：加上冒號比對，精準隔離 Agent 發言，防止 Agent_1 匹配到 Agent_11 的資料
            agent_speeches = [h for h in conversation_history if h.startswith(f"{name}:")]
            init_score = STANCE_SCORE_MAP[persona.initial_stance]
            initial_scores.append(init_score)

            # 🌟 修正 2：實施單盲盲測！徹底拿掉初始立場與初始分數，防範 Judge 模型產生預設偏誤污染
            analysis_prompt = f"""
            你是一位社會心理學家。請仔細閱讀參與者「{name}」在討論區的所有發言紀錄。
            
            該參與者的發言紀錄：
            {agent_speeches}

            請僅根據上述發言紀錄，客觀判斷其在討論結束時，對於「{TOPIC}」所展現的最終立場分數 (1.0 - 10.0 的浮點數)：
            - 1.0 - 3.0 代表偏向 Oppose(反對)
            - 5.0 代表 Neutral(中立)
            - 7.0 - 10.0 代表偏向 Support(支持)

            請嚴格以 JSON 格式回答，不要包含 markdown 標籤或任何其他文字：
            {{"final_score": 數字, "reason": "簡短理由"}}
            """

            if is_first_judge_call:
                print(f"首次喚醒 32B 判官模型，載入時間較長，預防性等待...")
                time.sleep(3) 
                is_first_judge_call = False

            try:
                raw_result = str(analyzer.invoke(analysis_prompt).content)

                json_str = re.search(r'\{.*\}', raw_result, re.DOTALL).group()
                res = json.loads(json_str)

                f_score = float(res['final_score'])
                neutral_point = 5.0
                
                # 1. 如果分數完全沒變，或是微幅震盪（差距小於等於 0.5）
                if abs(f_score - init_score) <= 0.5:
                    calculated_shift_type = "堅持"
                else:
                    # 2. 判斷是向中立點靠攏（從眾），還是遠離中立點（極化）
                    # 依據林憲聰論文定義：極化指擴大既存傾向，中立者若大動搖至極端則歸於動搖
                    init_distance_to_neutral = abs(init_score - neutral_point)
                    final_distance_to_neutral = abs(f_score - neutral_point)
                    
                    if final_distance_to_neutral < init_distance_to_neutral:
                        calculated_shift_type = "從眾"  # 變得更中立、更具共識
                    elif final_distance_to_neutral > init_distance_to_neutral:
                        if persona.initial_stance in ["Support", "Oppose"]:
                            calculated_shift_type = "極化"  # 有鮮明立場者往 1 或 10 的兩極端進一步放大
                        else:
                            calculated_shift_type = "動搖"  # 原本中立者被拉往某一極端
                    else:
                        calculated_shift_type = "立場轉變"

                final_scores.append(f_score)

                agent_details[name] = {
                    "initial_stance": persona.initial_stance,
                    "initial_score": init_score,
                    "final_score": f_score,
                    "shift_type": calculated_shift_type,
                    "reason": res['reason']
                }
            except Exception as parse_error:
                print(f"[{name}] Failed to parse judge output due to {parse_error}. Falling back to neutral score 5.0.")
                final_scores.append(5.0)
                agent_details[name] = {
                    "initial_stance": persona.initial_stance,
                    "initial_score": init_score,
                    "final_score": 5.0,
                    "shift_type": "Parsing_Failed",
                    "reason": raw_result
                }

        # 計算指標並透過持久化日誌儲存
        if len(final_scores) == len(initial_scores):
            metrics = calculate_metrics(initial_scores, final_scores)
            logger_service.save_experiment_result(metrics, agent_details, conversation_history)
        # 🌟 修正 3：加上安全網，若資料對齊失敗不容許靜悄悄跳過，必須發出警告
        else:
            print(f"⚠️ [Data Mismatch] Run #{run_id} alignment failed. Score lists length mismatch!")


# === 5. 多個 llm model 分群(目前只有一群)實驗進入點 ===
if __name__ == "__main__":
    script_identity = "matrix-run-group-a-optimized"

    print("="*60)
    print(f"Starting Highly Optimized Group A Multi-Model Matrix (6-Agent Config)")
    print("="*60)

    for target_model in GROUP_A_MODELS:
        print(f"\n[Matrix Schedular] Current Target Model Group: {target_model}")
        
        logger_service = SimulationLoggerService(
            topic=TOPIC,
            model_name=target_model,
            script_name=script_identity
        )

        # ----------------------------------------------------
        # 額外防禦：在載入對話模型前，確保 Judge 模型已經被清空
        # ----------------------------------------------------
        unload_ollama_model(JUDGE_MODEL)

        # ----------------------------------------------------
        # 步驟一：全速跑完該模型的 25 次對話，不切換 Judge
        # ----------------------------------------------------
        current_model_batch = []
        for run in range(1, REPETITIONS + 1):
            try:
                experiment_meta = run_dialogue_pipeline(target_model, current_run=run)
                current_model_batch.append(experiment_meta)
            except Exception as dialogue_error:
                print(f" [Dialogue Error] Run #{run} failed: {dialogue_error}. Skip.")
                continue
        
        # 卸除llm
        unload_ollama_model(target_model)

        # ----------------------------------------------------
        # 步驟二：載入 32B 判官，一口氣批次評分完 25 場實驗
        # ----------------------------------------------------
        if current_model_batch:
            try:
                evaluate_batch_experiments(target_model, current_model_batch, logger_service)
            except Exception as judge_error:
                print(f" [Critical Judge Error] Batch evaluation failed for {target_model}: {judge_error}")
            finally:
                # 該模型批次評分完，立刻踢出 32B Judge，為下一個循環釋放純淨 VRAM
                unload_ollama_model(JUDGE_MODEL)

    print("\n" + "="*60)
    print(f"Successfully completed all Pipeline Matrix Experiments for Group A!")
    print(f"All academic records are saved under: {script_identity}-result/")
    print("="*60)